# Step 4. Final Validation on a Random Sample

This notebook evaluates the finalized annotation procedure on a random sample of `n = 600` paragraphs. 

Workflow
1. Draw the random sample and generate model annotations.
2. Import independent labels from two human annotators.
3. Calculate intercoder reliability and resolve disagreements.
4. Create the final human labels and run the model again without access to them.
5. Compare model and human labels using accuracy, precision, recall, F1, Cohen's kappa, and a confusion matrix.

Final Evaluation

The final evaluation focuses on `NO_CRIME_FRAME` and `CRIME_FRAME_NEG`, since `CRIME_FRAME_POS` occurs too rarely for reliable evaluation.

## Imports and file paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import importlib
import krippendorff
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    cohen_kappa_score,
    precision_score,
    recall_score
)

In [2]:
load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO - check your .env file'}")
print(f"Host: {HOST}")
print(f"Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1
Model: ministral-3-14b


In [3]:
#for model validity check 
import annotation_setup
importlib.reload(annotation_setup)

from annotation_setup import (
    VALID_LABELS,
    instructions,
    reminder,
    annotate,
    parse_label,
    parse_explanation
)

print("Shared annotation setup loaded.")

Shared annotation setup loaded.


In [41]:
# ── Config ───────────────────────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Step 4 output files
STEP4_TESTSET_FOR_HUMAN_600 = RESULTS_DIR / "step4_testset_600_for_human_annotation.csv"
STEP4_HUMAN_HARDCASES_600 = "results/step4_human_hardcases_for_discussion_600.csv"
STEP4_HUMAN_COMPLETED_600 = "results/step4_human_completed_ground_truth_600.csv"
STEP4_HUMAN_COMPLETED_WITH_MODEL_600 = "results/step4_human_completed_ground_truth_600_with_model.csv"
STEP4_HUMAN_MODEL_COMPARISON_600 = "results/step4_human_model_comparison_600.csv"
STEP4_MODEL_HARDCASES_600 = "results/step4_model_hardcases_600.csv"
STEP4_EVALUATION_600 = "results/step4_evaluation_summary_600.csv"


In [7]:
# --- Config ---
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

SAMPLE_SIZE_2_1 = 600
RANDOM_SEED = 2

# Saving of full run result
OUTPUT_2_1 = RESULTS_DIR / f"annotation_step2_1_seed{RANDOM_SEED}_n{SAMPLE_SIZE_2_1}.csv"

UNCLEAR_LABELS = ["UNCLEAR"]

# Manual testset later (n=(2x 250 batches))
MANUAL_TESTSET = RESULTS_DIR / "manual_testset.csv"
# ---

df = pd.read_csv("dataset/all_multi_paragraphs_2022_2026.csv")
sample_2_1 = df.sample(n=SAMPLE_SIZE_2_1, random_state=RANDOM_SEED).copy().reset_index(drop=True)

print(f"Dataset loaded {len(df)} rows total")
print(f"Sample size {len(sample_2_1)} rows")
print("Running annotation at temperature=0.0 (deterministic)\n")

Dataset loaded 659895 rows total
Sample size 600 rows
Running annotation at temperature=0.0 (deterministic)



In [8]:
results_2_1 = []

for i, row in sample_2_1.iterrows():
    raw = annotate(
        text=str(row["text_block"]),
        instructions=instructions,
        reminder=reminder,
        api_key=api_key,
        HOST=HOST,
        MODEL=MODEL,
        temperature=0.0
    )
    label = parse_label(raw)
    explanation = parse_explanation(raw)

    results_2_1.append({
        "article_id":    row["article_id"],
        "par_index":     row["par_index"],
        "group":         row["group"],
        "text_block_en": row["text_block"],
        "raw_output":    raw,
        "label":         label,
        "explanation":   explanation
    })

    if (i + 1) % 10 == 0:
        n_neg = sum(1 for r in results_2_1 if r["label"] == "CRIME_FRAME_NEG")
        n_pos = sum(1 for r in results_2_1 if r["label"] == "CRIME_FRAME_POS")
        n_unclear = sum(1 for r in results_2_1 if r["label"] == "UNCLEAR")

        print(
            f"  [{i+1}/{len(sample_2_1)}] done "
            f"NEG: {n_neg}, POS: {n_pos}, UNCLEAR: {n_unclear}"
        )

annotation_2_1 = pd.DataFrame(results_2_1)

# Save full Step 2.1 output only
annotation_2_1.to_csv(OUTPUT_2_1, index=False)

print(f"\nSaved Step 2.1 to {OUTPUT_2_1}")
print("\nDone!")
print(annotation_2_1["label"].value_counts())

crime_total = annotation_2_1["label"].isin(["CRIME_FRAME_NEG", "CRIME_FRAME_POS"]).sum()

print(f"\nAny CRIME_FRAME rate: {crime_total / len(annotation_2_1) * 100:.1f}%")
print(f"CRIME_FRAME_NEG rate: {(annotation_2_1['label'] == 'CRIME_FRAME_NEG').mean() * 100:.1f}%")
print(f"CRIME_FRAME_POS rate: {(annotation_2_1['label'] == 'CRIME_FRAME_POS').mean() * 100:.1f}%")
print(f"NO_CRIME_FRAME rate: {(annotation_2_1['label'] == 'NO_CRIME_FRAME').mean() * 100:.1f}%")
print(f"UNCLEAR rate: {(annotation_2_1['label'] == 'UNCLEAR').mean() * 100:.1f}%")

  [10/600] done NEG: 0, POS: 0, UNCLEAR: 0
  [20/600] done NEG: 0, POS: 0, UNCLEAR: 0
  [30/600] done NEG: 0, POS: 0, UNCLEAR: 0
  [40/600] done NEG: 0, POS: 0, UNCLEAR: 0
  [50/600] done NEG: 1, POS: 0, UNCLEAR: 0
  [60/600] done NEG: 1, POS: 0, UNCLEAR: 0
  [70/600] done NEG: 3, POS: 0, UNCLEAR: 0
  [80/600] done NEG: 3, POS: 0, UNCLEAR: 0
  [90/600] done NEG: 3, POS: 0, UNCLEAR: 0
  [100/600] done NEG: 3, POS: 0, UNCLEAR: 0
  [110/600] done NEG: 3, POS: 0, UNCLEAR: 0
  [120/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [130/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [140/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [150/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [160/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [170/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [180/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [190/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [200/600] done NEG: 4, POS: 0, UNCLEAR: 0
  [210/600] done NEG: 5, POS: 0, UNCLEAR: 0
  [220/600] done NEG: 7, POS: 0, UNCLEAR: 0
  [230/600] done NEG: 7, POS: 0, UNCLEAR:

In [31]:
# --- Config ---
SAMPLE_SIZE_2_2 = 600
N_RUNS = 10
TEMPERATURE = 0.7

# Seed only for selecting NO_CRIME_FRAME rows!
NO_CRIME_FILL_SEED = 100

HARD_CASES_FRAME = RESULTS_DIR / "hard_cases.csv"
OUTPUT_2_2 = RESULTS_DIR / f"annotation_step2_2_fillseed{NO_CRIME_FILL_SEED}_n{SAMPLE_SIZE_2_2}_runs{N_RUNS}.csv"
# ---

# Include all CRIME_FRAME_NEG, CRIME_FRAME_POS, and UNCLEAR cases from Step 2.1
interesting_cases = annotation_2_1[
    annotation_2_1["label"].isin(["CRIME_FRAME_NEG", "CRIME_FRAME_POS", "UNCLEAR"])
].copy()

# Fill remaining slots with NO_CRIME_FRAME cases
n_fill = SAMPLE_SIZE_2_2 - len(interesting_cases)

if n_fill > 0:
    no_crime_candidates = annotation_2_1[
        annotation_2_1["label"] == "NO_CRIME_FRAME"
    ].copy()

    no_crime_fill = no_crime_candidates.sample(
        n=min(n_fill, len(no_crime_candidates)),
        random_state=NO_CRIME_FILL_SEED
    ).copy()

    sample_2_2 = pd.concat([interesting_cases, no_crime_fill], ignore_index=True)
else:
    no_crime_fill = pd.DataFrame()
    sample_2_2 = interesting_cases.copy()

sample_2_2 = sample_2_2.drop_duplicates(
    subset=["article_id", "par_index"]
).reset_index(drop=True)

print(f"Interesting cases included: {len(interesting_cases)}")
print(f"NO_CRIME_FRAME fill cases included: {len(no_crime_fill)}")
print(f"Sample size: {len(sample_2_2)} rows")
print(f"Runs per paragraph: {N_RUNS}")
print(f"Total API calls: {len(sample_2_2) * N_RUNS}")
print(f"Temperature: {TEMPERATURE}")
print(sample_2_2["label"].value_counts())

Interesting cases included: 25
NO_CRIME_FRAME fill cases included: 575
Sample size: 600 rows
Runs per paragraph: 10
Total API calls: 6000
Temperature: 0.7
label
NO_CRIME_FRAME     575
CRIME_FRAME_NEG     25
Name: count, dtype: int64


In [34]:
#for new random testset n=600 (run 10x modellabel on each cell) 
from pathlib import Path
import pandas as pd

# Config
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Input file
STEP2_600_INPUT = RESULTS_DIR / "annotation_step2_2_fillseed100_n600_runs10.csv"

# Output file
STEP4_TESTSET_600_FOR_HUMAN_ANNOTATION = RESULTS_DIR / "step4_testset_600_for_human_annotation.csv"

# Load existing 600-row CSV
step4_testset = pd.read_csv(STEP2_600_INPUT)

# Add testset_id if it does not exist yet
if "testset_id" not in step4_testset.columns:
    step4_testset.insert(0, "testset_id", range(1, len(step4_testset) + 1))

# Create human annotation export
human_export = step4_testset[
    ["testset_id", "article_id", "par_index", "group", "text_block_en"]
].copy()

human_export["label_robin"] = ""
human_export["label_marmee"] = ""

human_export["notes_robin"] = ""
human_export["notes_marmee"] = ""

# Save human annotation file
human_export.to_csv(STEP4_TESTSET_600_FOR_HUMAN_ANNOTATION, index=False)

print(f"Human annotation file saved to {STEP4_TESTSET_600_FOR_HUMAN_ANNOTATION}")

human_export.head()

Human annotation file saved to results/step4_testset_600_for_human_annotation.csv


,testset_id,article_id,par_index,group,text_block_en,label_robin,label_marmee,notes_robin,notes_marmee
0,1,637a6c60130808d0340ec37f,9,KURD,Die regierungsnahe Nachrichtenagentur Tasnim s...,,,,
1,2,Spiegel_added_21a6799f309e34741e7f7e078ee636e4...,4,AUT,"22.11.2022, 16.52 Uhr\n\nEs sind keine Unbekan...",,,,
2,3,632712bb3dd49b919c23a58a,5,POC,Mz xvs wziu 100 Dfbchschmcrog ifh Ifukvrcvuüzp...,,,,
3,4,637677e1effe4afbd70001da,12,RUS,Die Angeklagten erschienen nicht vor Gericht\n...,,,,
4,5,6327127f3dd49b919c0b142d,2,MUSL,Widerspruch und Spaß werden nicht geduldet – e...,,,,


## Step 4.1: Create Human Consensus Labels + Assess Human Intercoder Reliability
Both human annotators code the random sample of `n = 600` paragraphs independently. When we assign the same label, that label becomes the human consensus label. When we disagree, the paragraph is flagged as a human hardcase, reviewed together, and assigned a final consensus label in `final_human_label`.

Human reliability is calculated before disagreements are resolved. We report raw percentage agreement and Krippendorff's alpha to assess how consistently both annotators apply the finalized annotation instructions.


In [62]:
# ── Create human consensus labels and check reliability for testset n=600──────

human_annotation = pd.read_csv(STEP4_TESTSET_FOR_HUMAN_600)

annotator_cols = ["label_robin", "label_marmee"]

# Clean labels
for col in annotator_cols:
    human_annotation[col] = human_annotation[col].astype(str).str.strip()
    human_annotation[col] = human_annotation[col].replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan
    })

# Check for invalid labels or typos
for col in annotator_cols:
    invalid = human_annotation[
        human_annotation[col].notna() &
        ~human_annotation[col].isin(VALID_LABELS)
    ]

    if len(invalid) > 0:
        print(f"Warning: {col} has invalid labels:")
        display(invalid[["testset_id", col]].head(20))


def get_consensus_label(row):
    robin = row["label_robin"]
    marmee = row["label_marmee"]

    if pd.isna(robin) or pd.isna(marmee):
        return np.nan

    if robin == marmee:
        return robin

    return "NO_HUMAN_CONSENSUS"


def get_human_agreement(row):
    robin = row["label_robin"]
    marmee = row["label_marmee"]

    if pd.isna(robin) or pd.isna(marmee):
        return np.nan

    return 1.0 if robin == marmee else 0.0


def get_n_annotators(row):
    return sum(pd.notna(row[col]) for col in annotator_cols)


human_annotation["n_human_annotators"] = human_annotation.apply(get_n_annotators, axis=1)
human_annotation["human_consensus_label"] = human_annotation.apply(get_consensus_label, axis=1)
human_annotation["human_agreement"] = human_annotation.apply(get_human_agreement, axis=1)


human_hard_cases = human_annotation[
    human_annotation["human_consensus_label"] == "NO_HUMAN_CONSENSUS"
].copy()

if "final_human_label" not in human_hard_cases.columns:
    human_hard_cases["final_human_label"] = ""
else:
    human_hard_cases["final_human_label"] = (
        human_hard_cases["final_human_label"]
        .fillna("")
    )

human_hard_cases.to_csv(
    STEP4_HUMAN_HARDCASES_600,
    index=False
)

print("=== Human consensus creation ===")

print("Number of annotators per row:")
print(human_annotation["n_human_annotators"].value_counts(dropna=False).sort_index())

print("\nConsensus label distribution:")
print(human_annotation["human_consensus_label"].value_counts(dropna=False))

double_coded = human_annotation[human_annotation["n_human_annotators"] == 2].copy()

n_total = len(double_coded)
n_agree = double_coded["human_consensus_label"].isin(VALID_LABELS).sum()
n_disagree = (double_coded["human_consensus_label"] == "NO_HUMAN_CONSENSUS").sum()

percent_agreement = n_agree / n_total if n_total > 0 else np.nan
percent_disagreement = n_disagree / n_total if n_total > 0 else np.nan

print("\n=== Human reliability summary ===")
print(f"Double-coded rows: {n_total}")
print(f"Rows with agreement: {n_agree} ({percent_agreement * 100:.1f}%)")
print(f"Rows without agreement: {n_disagree} ({percent_disagreement * 100:.1f}%)")

print("\nHuman agreement distribution among double-coded rows:")
print(double_coded["human_agreement"].describe().round(2))

# Cohen's kappa and Krippendorff's alpha between the two annotators
pair_data = human_annotation[
    human_annotation["label_robin"].isin(VALID_LABELS) &
    human_annotation["label_marmee"].isin(VALID_LABELS)
].copy()

if len(pair_data) > 0:
    pair_accuracy = accuracy_score(pair_data["label_robin"], pair_data["label_marmee"])
    pair_kappa = cohen_kappa_score(
        pair_data["label_robin"],
        pair_data["label_marmee"],
        labels=VALID_LABELS
    )
    
    # Prepare data structure for Krippendorff (a list of lists/matrix where rows=coders, columns=items)
    reliability_data = [
        pair_data["label_robin"].tolist(),
        pair_data["label_marmee"].tolist()
    ]
    pair_alpha = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement="nominal")
else:
    pair_accuracy = np.nan
    pair_kappa = np.nan
    pair_alpha = np.nan

print("\n=== Pairwise human reliability ===")
print(f"Overlapping valid annotations: {len(pair_data)}")
print(f"Percent agreement: {pair_accuracy * 100:.1f}%")
print(f"Cohen's kappa: {pair_kappa:.3f}") # can be excluded bc use of Krippendorffs 
print(f"Krippendorff's alpha (nominal): {pair_alpha:.3f}")

print(f"\nHuman hard cases without consensus: {len(human_hard_cases)}")
print(f"Saved human hard cases to {STEP4_HUMAN_HARDCASES_600}")

=== Human consensus creation ===
Number of annotators per row:
n_human_annotators
2    600
Name: count, dtype: int64

Consensus label distribution:
human_consensus_label
NO_CRIME_FRAME        550
NO_HUMAN_CONSENSUS     27
CRIME_FRAME_NEG        23
Name: count, dtype: int64

=== Human reliability summary ===
Double-coded rows: 600
Rows with agreement: 573 (95.5%)
Rows without agreement: 27 (4.5%)

Human agreement distribution among double-coded rows:
count    600.00
mean       0.96
std        0.21
min        0.00
25%        1.00
50%        1.00
75%        1.00
max        1.00
Name: human_agreement, dtype: float64

=== Pairwise human reliability ===
Overlapping valid annotations: 600
Percent agreement: 95.5%
Cohen's kappa: 0.609
Krippendorff's alpha (nominal): 0.607

Human hard cases without consensus: 27
Saved human hard cases to results/step4_human_hardcases_for_discussion_600.csv


In [64]:
# ── After solving hard cases: Create final human completed ground truth for n=600 ──

human_annotation = pd.read_csv(STEP4_TESTSET_FOR_HUMAN_600)
human_hard_cases = pd.read_csv(STEP4_HUMAN_HARDCASES_600)

human_annotation.columns = human_annotation.columns.str.strip()
human_hard_cases.columns = human_hard_cases.columns.str.strip()

annotator_cols = ["label_robin", "label_marmee"]


def clean_label_series(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "None": np.nan,
            "NONE": np.nan
        })
    )


def normalize_testset_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype(str).replace("<NA>", np.nan)


# Drop completely empty rows from the annotation file
human_annotation = human_annotation.dropna(how="all").copy()

# Clean testset_id
human_annotation["testset_id"] = normalize_testset_id(human_annotation["testset_id"])
human_hard_cases["testset_id"] = normalize_testset_id(human_hard_cases["testset_id"])

# Drop rows without a real testset_id
human_annotation = human_annotation[human_annotation["testset_id"].notna()].copy()
human_hard_cases = human_hard_cases[human_hard_cases["testset_id"].notna()].copy()

# Clean annotator labels
for df_tmp in [human_annotation, human_hard_cases]:
    for col in annotator_cols:
        if col in df_tmp.columns:
            df_tmp[col] = clean_label_series(df_tmp[col])

# Clean final hard case labels
if "final_human_label" not in human_hard_cases.columns:
    raise ValueError(
        f"'final_human_label' column is missing in {STEP4_HUMAN_HARDCASES_600}"
    )

human_hard_cases["final_human_label"] = clean_label_series(
    human_hard_cases["final_human_label"]
)

# Recreate consensus labels
human_annotation["human_consensus_label"] = human_annotation.apply(
    get_consensus_label,
    axis=1
)

# Keep only solved hard cases
resolved_hard_cases = human_hard_cases[
    human_hard_cases["final_human_label"].isin(VALID_LABELS)
][["testset_id", "final_human_label"]].copy()

resolved_hard_cases = resolved_hard_cases.drop_duplicates(
    subset=["testset_id"],
    keep="last"
)

# Map manually resolved hard case labels into full file
final_label_map = resolved_hard_cases.set_index("testset_id")["final_human_label"]

human_completed = human_annotation.copy()

human_completed["final_human_label_from_hardcases"] = (
    human_completed["testset_id"].map(final_label_map)
)

# Use hardcase final_human_label first, otherwise use consensus
human_completed["final_human_label"] = human_completed[
    "final_human_label_from_hardcases"
].fillna(
    human_completed["human_consensus_label"]
)

# Keep only rows with valid final labels
human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy()

human_completed = human_completed.drop(
    columns=["final_human_label_from_hardcases"]
)

human_completed.to_csv(STEP4_HUMAN_COMPLETED_600, index=False)

print(f"Saved completed human ground truth to {STEP4_HUMAN_COMPLETED_600}")
print(f"Rows in original 600 file after dropping empty rows: {len(human_annotation)}")
print(f"Rows with solved hard cases: {len(resolved_hard_cases)}")
print(f"Rows in completed ground truth: {len(human_completed)}")

print("\nFinal human label distribution:")
print(human_completed["final_human_label"].value_counts())

missing_final = len(human_annotation) - len(human_completed)

print(f"\nRows without final valid human label: {missing_final}")

if missing_final > 0:
    unresolved = human_annotation.merge(
        human_completed[["testset_id"]],
        on="testset_id",
        how="left",
        indicator=True
    )

    unresolved = unresolved[
        unresolved["_merge"] == "left_only"
    ].drop(columns=["_merge"])

    print("\nUnresolved rows preview:")
    display(
        unresolved[
            ["testset_id", "label_robin", "label_marmee", "human_consensus_label"]
        ].head(30)
    )

Saved completed human ground truth to results/step4_human_completed_ground_truth_600.csv
Rows in original 600 file after dropping empty rows: 600
Rows with solved hard cases: 27
Rows in completed ground truth: 600

Final human label distribution:
final_human_label
NO_CRIME_FRAME     574
CRIME_FRAME_NEG     26
Name: count, dtype: int64

Rows without final valid human label: 0


## 4.3 Rerun Model Labels on Human Completed Cases

In [55]:
# Rerun model and overwrite model columns for completed n=600

human_completed = pd.read_csv(STEP4_HUMAN_COMPLETED_600)
human_completed.columns = human_completed.columns.str.strip()

def normalize_testset_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype(str).replace("<NA>", np.nan)

human_completed["testset_id"] = normalize_testset_id(human_completed["testset_id"])
human_completed = human_completed[human_completed["testset_id"].notna()].copy()

human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

print(f"Human completed rows with valid final label: {len(human_completed)}")
print(f"Running fresh model labels with model: {MODEL}")

required_cols = [
    "testset_id",
    "text_block_en",
    "final_human_label"
]

missing_cols = [
    col for col in required_cols
    if col not in human_completed.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns in {STEP3_HUMAN_COMPLETED_600}: {missing_cols}"
    )

for col in ["model_label", "model_explanation", "model_raw_output", "model"]:
    if col not in human_completed.columns:
        human_completed[col] = pd.NA

fresh_labels = {}

for i, row in human_completed.iterrows():
    raw = annotate(
        text=str(row["text_block_en"]),
        instructions=instructions,
        reminder=reminder,
        api_key=api_key,
        HOST=HOST,
        MODEL=MODEL,
        temperature=0.0
    )

    fresh_labels[row["testset_id"]] = {
        "model_label": parse_label(raw),
        "model_explanation": parse_explanation(raw),
        "model_raw_output": raw,
        "model": MODEL
    }

    if (i + 1) % 10 == 0 or (i + 1) == len(human_completed):
        print(f"[{i + 1}/{len(human_completed)}] done")

for testset_id, values in fresh_labels.items():
    mask = human_completed["testset_id"] == testset_id

    human_completed.loc[mask, "model_label"] = values["model_label"]
    human_completed.loc[mask, "model_explanation"] = values["model_explanation"]
    human_completed.loc[mask, "model_raw_output"] = values["model_raw_output"]
    human_completed.loc[mask, "model"] = values["model"]

human_completed.to_csv(STEP4_HUMAN_COMPLETED_WITH_MODEL_600, index=False)

print(f"\nSaved completed new 250 file with fresh model labels to {STEP4_HUMAN_COMPLETED_WITH_MODEL_600}")

print("\nUpdated model label distribution:")
print(
    human_completed["model_label"].value_counts(dropna=False)
)

print("\nFinal human label distribution:")
print(
    human_completed["final_human_label"].value_counts(dropna=False)
)

Human completed rows with valid final label: 600
Running fresh model labels with model: ministral-3-14b
[10/600] done
[20/600] done
[30/600] done
[40/600] done
[50/600] done
[60/600] done
[70/600] done
[80/600] done
[90/600] done
[100/600] done
[110/600] done
[120/600] done
[130/600] done
[140/600] done
[150/600] done
[160/600] done
[170/600] done
[180/600] done
[190/600] done
[200/600] done
[210/600] done
[220/600] done
[230/600] done
[240/600] done
[250/600] done
[260/600] done
[270/600] done
[280/600] done
[290/600] done
[300/600] done
[310/600] done
[320/600] done
[330/600] done
[340/600] done
[350/600] done
[360/600] done
[370/600] done
[380/600] done
[390/600] done
[400/600] done
[410/600] done
[420/600] done
[430/600] done
[440/600] done
[450/600] done
[460/600] done
[470/600] done
[480/600] done
[490/600] done
[500/600] done
[510/600] done
[520/600] done
[530/600] done
[540/600] done
[550/600] done
[560/600] done
[570/600] done
[580/600] done
[590/600] done
[600/600] done

Save

## Step 4.3: Assess Model Validity

The majority-vote model labels are compared with the final human consensus labels. Model validity is evaluated using precision, recall, macro-F1, weighted-F1, and Cohen's kappa. The final evaluation focuses on the distinction between `NO_CRIME_FRAME` and `CRIME_FRAME_NEG`, because `CRIME_FRAME_POS` is too rare for reliable evaluation.

In [60]:
#Validity checks against human ground truth for n=600 excluded crime_frame_pos


# Labels used for this binary evaluation
EVAL_LABELS = [
    "NO_CRIME_FRAME",
    "CRIME_FRAME_NEG"
]


# Load completed human annotations with fresh model labels
comparison = pd.read_csv(STEP4_HUMAN_COMPLETED_WITH_MODEL_600)

# Remove accidental whitespace from column names
comparison.columns = comparison.columns.str.strip()


def clean_label_series(series):
    """
    Clean annotation labels and convert empty values to missing values.
    """
    return (
        series
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "NONE": pd.NA
        })
    )


def normalize_testset_id(series):
    """
    Convert testset_id values to normalized integer-like strings.
    """
    numeric = pd.to_numeric(series, errors="coerce")

    return (
        numeric
        .astype("Int64")
        .astype("string")
        .replace("<NA>", pd.NA)
    )


# Required columns
required_cols = [
    "testset_id",
    "final_human_label",
    "model_label"
]

missing_cols = [
    col
    for col in required_cols
    if col not in comparison.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns in "
        f"{STEP4_HUMAN_COMPLETED_WITH_MODEL_600}: "
        f"{missing_cols}"
    )


# Clean identifiers and labels
comparison["testset_id"] = normalize_testset_id(
    comparison["testset_id"]
)

comparison["final_human_label"] = clean_label_series(
    comparison["final_human_label"]
)

comparison["model_label"] = clean_label_series(
    comparison["model_label"]
)


# Remove rows without a valid testset_id
comparison = (
    comparison[
        comparison["testset_id"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


# Keep rows with a valid final human label
comparison = (
    comparison[
        comparison["final_human_label"].isin(EVAL_LABELS)
    ]
    .copy()
    .reset_index(drop=True)
)


# Mark whether the model label is valid for this evaluation
comparison["model_valid_label"] = (
    comparison["model_label"].isin(EVAL_LABELS)
)


# Human and model agreement
comparison["human_model_agree"] = (
    comparison["final_human_label"]
    == comparison["model_label"]
)

comparison["agreement_status"] = np.where(
    comparison["human_model_agree"],
    "AGREE",
    "DISAGREE"
)


# Arrange important columns first
preferred_columns = [
    "testset_id",
    "group",
    "text_block_en",
    "label_robin",
    "label_marmee",
    "human_consensus_label",
    "final_human_label",
    "model",
    "model_label",
    "model_explanation",
    "model_raw_output",
    "model_valid_label",
    "human_model_agree",
    "agreement_status",
    "article_id",
    "par_index"
]

available_preferred_columns = [
    col
    for col in preferred_columns
    if col in comparison.columns
]

remaining_columns = [
    col
    for col in comparison.columns
    if col not in available_preferred_columns
]

comparison = comparison[
    available_preferred_columns + remaining_columns
].copy()


# Save complete comparison file
comparison.to_csv(
    STEP4_HUMAN_MODEL_COMPARISON_600,
    index=False
)

print(
    f"Saved human model comparison to "
    f"{STEP4_HUMAN_MODEL_COMPARISON_600}"
)


# Agreement counts
print("\nAgreement counts:")

print(
    comparison["agreement_status"]
    .value_counts(dropna=False)
)


# Save model disagreements
model_hardcases = comparison[
    comparison["agreement_status"] == "DISAGREE"
].copy()

model_hardcases.to_csv(
    STEP4_MODEL_HARDCASES_600,
    index=False
)

print(
    f"\nSaved model hardcases to "
    f"{STEP4_MODEL_HARDCASES_600}"
)

print(
    f"Number of model hardcases: "
    f"{len(model_hardcases)}"
)


# Keep only rows where both labels are valid
eval_data = comparison[
    comparison["final_human_label"].isin(EVAL_LABELS)
    & comparison["model_label"].isin(EVAL_LABELS)
].copy()


print("\nModel validity against human ground truth")

print(
    f"Rows used for model evaluation: "
    f"{len(eval_data)}"
)

print(
    f"Rows excluded: "
    f"{len(comparison) - len(eval_data)}"
)


# Label distributions
print("\nHuman label distribution:")

print(
    eval_data["final_human_label"]
    .value_counts(dropna=False)
)


print("\nModel label distribution:")

print(
    eval_data["model_label"]
    .value_counts(dropna=False)
)


if len(eval_data) == 0:
    raise ValueError(
        "No valid rows available for evaluation. "
        "Check final_human_label and model_label values."
    )


# Ground truth and predictions
y_true = eval_data["final_human_label"]
y_pred = eval_data["model_label"]


# Overall accuracy
accuracy = accuracy_score(
    y_true,
    y_pred
)


# Macro metrics across the two observed classes
macro_precision = precision_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    average="macro",
    zero_division=0
)


# Weighted F1
weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    average="weighted",
    zero_division=0
)


# Cohen's kappa
kappa = cohen_kappa_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS
)


# Positive class metrics
positive_precision = precision_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    pos_label="CRIME_FRAME_NEG",
    average="binary",
    zero_division=0
)

positive_recall = recall_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    pos_label="CRIME_FRAME_NEG",
    average="binary",
    zero_division=0
)

positive_f1 = f1_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    pos_label="CRIME_FRAME_NEG",
    average="binary",
    zero_division=0
)


# Print summary metrics
print("\nSummary metrics:")

print(f"Accuracy: {accuracy:.3f}")
print(f"Macro precision: {macro_precision:.3f}")
print(f"Macro recall: {macro_recall:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")
print(f"Cohen's kappa: {kappa:.3f}")

print("\nCRIME_FRAME_NEG metrics:")

print(f"Precision: {positive_precision:.3f}")
print(f"Recall: {positive_recall:.3f}")
print(f"F1: {positive_f1:.3f}")


# Confusion matrix
print("\nConfusion matrix:")

conf_matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=EVAL_LABELS
)

conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=[
        f"human_{label}"
        for label in EVAL_LABELS
    ],
    columns=[
        f"model_{label}"
        for label in EVAL_LABELS
    ]
)

display(conf_matrix_df)


# Classification report
print("\nClassification report:")

print(
    classification_report(
        y_true,
        y_pred,
        labels=EVAL_LABELS,
        target_names=EVAL_LABELS,
        zero_division=0
    )
)


# Determine model name for the evaluation summary
if "MODEL" in globals():
    evaluation_model = MODEL

elif (
    "model" in comparison.columns
    and comparison["model"].notna().any()
):
    evaluation_model = (
        comparison["model"]
        .dropna()
        .iloc[0]
    )

else:
    evaluation_model = None


# Create evaluation summary
evaluation_summary = pd.DataFrame([{
    "model": evaluation_model,
    "evaluation_labels": "|".join(EVAL_LABELS),
    "n_total_human_completed": len(comparison),
    "n_eval": len(eval_data),
    "n_excluded": len(comparison) - len(eval_data),
    "n_agree": int(
        comparison["human_model_agree"].sum()
    ),
    "n_disagree": int(
        (~comparison["human_model_agree"]).sum()
    ),
    "agreement_rate": (
        comparison["human_model_agree"].mean()
    ),
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "crime_frame_neg_precision": positive_precision,
    "crime_frame_neg_recall": positive_recall,
    "crime_frame_neg_f1": positive_f1,
    "cohens_kappa": kappa
}])


# Save evaluation summary
evaluation_summary.to_csv(
    STEP4_EVALUATION_600,
    index=False
)

print(
    f"\nSaved evaluation summary to "
    f"{STEP4_EVALUATION_600}"
)

Saved human model comparison to results/step4_human_model_comparison_600.csv

Agreement counts:
agreement_status
AGREE       592
DISAGREE      8
Name: count, dtype: int64

Saved model hardcases to results/step4_model_hardcases_600.csv
Number of model hardcases: 8

Model validity against human ground truth
Rows used for model evaluation: 600
Rows excluded: 0

Human label distribution:
final_human_label
NO_CRIME_FRAME     574
CRIME_FRAME_NEG     26
Name: count, dtype: Int64

Model label distribution:
model_label
NO_CRIME_FRAME     574
CRIME_FRAME_NEG     26
Name: count, dtype: Int64

Summary metrics:
Accuracy: 0.987
Macro precision: 0.920
Macro recall: 0.920
Macro F1: 0.920
Weighted F1: 0.987
Cohen's kappa: 0.839

CRIME_FRAME_NEG metrics:
Precision: 0.846
Recall: 0.846
F1: 0.846

Confusion matrix:


,model_NO_CRIME_FRAME,model_CRIME_FRAME_NEG
human_NO_CRIME_FRAME,570,4
human_CRIME_FRAME_NEG,4,22



Classification report:
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.99      0.99      0.99       574
CRIME_FRAME_NEG       0.85      0.85      0.85        26

       accuracy                           0.99       600
      macro avg       0.92      0.92      0.92       600
   weighted avg       0.99      0.99      0.99       600


Saved evaluation summary to results/step4_evaluation_summary_600.csv
